##### Copyright 2026 Google LLC.

In [24]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [25]:
%pip install -U -q "google-genai>=2.9.0"

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [26]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [27]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown
from google.colab import userdata

# Safely load the key if it hasn't been defined yet
try:
    GEMINI_API_KEY
except NameError:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [28]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is Paris.

In [29]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChdhSFBHYXBiMElkLWIxTWtQdXFQaXNBOBIXYUhQR2FwYjBJZC1iMU1rUHVxUGlzQTg
Environment ID: 2657e2b6ab957fff00b6be4b5a11b8d0


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [30]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have recorded your name and role in `knowledge.md`.

In [31]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** Based on the knowledge base in `knowledge.md`:

- **Name:** Alice
- **Role:** Software Engineer

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [32]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I created a Python script (`fibonacci.py`) to compute the first 20 Fibonacci numbers:

```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "__main__":
    n = 20
    fib_numbers = fibonacci(n)
    print(f"First {n} Fibonacci numbers:")
    for idx, num in enumerate(fib_numbers, 1):
        print(f"{idx}: {num}")
```

### Output:

```
First 20 Fibonacci numbers:
1: 0
2: 1
3: 1
4: 2
5: 3
6: 5
7: 8
8: 13
9: 21
10: 34
11: 55
12: 89
13: 144
14: 233
15: 377
16: 610
17: 987
18: 1597
19: 2584
20: 4181
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [33]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [thought] ---

--- Step 1 [function_call] ---
  Tool: write_file
  Args: {'path': '/tmp/fibonacci.py', 'toolAction': 'Writing Fibonacci script', 'toolSummary': 'Fibonacci script creation', 'explanation': 'Wrote Python script for computing Fibonacci numbers.', 'content': 'def fibonacci(n):\n    fib_series = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_series.a

--- Step 2 [function_result] ---
  Tool: write_file

--- Step 3 [code_execution_call] ---

--- Step 4 [code_execution_result] ---

--- Step 5 [model_output] ---
  Text: I created a Python script (`fibonacci.py`) to compute the first 20 Fibonacci numbers:

```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "__main__":
    n = 20
    fib_numb



### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [34]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

The top three news stories regarding Google this week are:

---

### 1. Google Signs Landmark 20-Year Nuclear Energy Deal with Constellation Energy
* **Summary**: On October 6, Google and Constellation Energy announced a major 20-year clean energy agreement to bring 890 megawatts (MW) of new nuclear power capacity to the PJM grid [1, 2]. The deal supports efficiency and output upgrades across 11 nuclear units in Illinois, Pennsylvania, and New Jersey to help satisfy surging energy demands driven by AI infrastructure [2]. In addition, Constellation will deploy Google Cloud and Gemini Enterprise as part of an expanded partnership to optimize energy grid operations [2].

---

### 2. Finland Orders Halt to Google AI Data Center Construction Over Environmental Concerns
* **Summary**: Finnish environmental regulators ordered Google’s subsidiary, Tuike Finland, to halt construction and preparatory site work at two planned data center locations in Muhos and Kajaani by October 23 [3, 4]. The authorities intervened after discovering that extensive forest clearing, grading, and site preparation had begun prior to completing mandatory environmental impact assessments (EIAs) [3]. The planned projects are part of Google's €13 billion ($14.6 billion) infrastructure investment program in Finland [3].

---

### 3. Google Faces £1.2 Billion ($1.6B) UK Consumer Class Action Trial Over Play Store Fees
* **Summary**: On October 6, a major collective lawsuit commenced at the UK Competition Appeal Tribunal in London on behalf of approximately 20 million British consumers [5, 6]. The class action alleges that Google abused its market dominance by charging app developers a 30% commission fee between 2015 and 2026, which was subsequently passed on to consumers in the form of inflated app and in-app purchase prices [5, 6]. Google has denied any anticompetitive conduct and called for the claims to be dismissed [5].

---

### Sources
* [1] [Market Index: Paladin, NexGen and Boss rally after Google's 20-year nuclear deal](https://www.marketindex.com.au/news/paladin-nexgen-and-boss-rally-after-googles-20-year-nuclear-deal)
* [2] [Constellation Energy: Google and Constellation Announce Landmark Agreement to Bring 890 MW of New Nuclear Capacity](https://www.constellationenergy.com/news/2026/10/google-and-constellation-announce-landmark-agreement-to-bring-890-mw-of-new-nuclear-capacity-to-pjm-grid.html)
* [3] [Al Jazeera: Finland orders halt to work on Google data centre sites](https://www.aljazeera.com/news/2026/10/7/finland-orders-halt-to-work-on-google-data-sites-over-environment-concerns)
* [4] [Techmeme: Finland orders Google subsidiary Tuike Finland Oy to halt data center construction](https://www.techmeme.com/261007/p9)
* [5] [Insurance Journal: Google Accused of Overcharging 20 Million Consumers in UK Class Action](https://www.insurancejournal.com/news/international/2026/10/07/888267.htm)
* [6] [The Star: Google accused of overcharging 20 million consumers in UK class action](https://www.thestar.com.my/tech/tech-news/2026/10/07/google-accused-of-overcharging-20-million-consumers-in-uk-class-action)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [35]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created `analysis.py` and executed it.

### Script Content (`analysis.py`)

```python
import random
import statistics

def main():
    # Set seed for reproducibility or generate random numbers
    random.seed(42)
    numbers = [random.uniform(1.0, 100.0) for _ in range(50)]

    # Compute statistics
    mean_val = statistics.mean(numbers)
    median_val = statistics.median(numbers)
    stdev_val = statistics.stdev(numbers)

    print("Generated 50 random numbers between 1.0 and 100.0.")
    print(f"Mean:               {mean_val:.4f}")
    print(f"Median:             {median_val:.4f}")
    print(f"Standard Deviation: {stdev_val:.4f}")

if __name__ == "__main__":
    main()
```

### Execution Output

```
Generated 50 random numbers between 1.0 and 100.0.
Mean:               45.6177
Median:             46.9002
Standard Deviation: 29.0356
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [36]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on `data.csv`, here is the breakdown of the scores:

- **Charlie**: 95
- **Alice**: 92
- **Eve**: 91
- **Diana**: 88
- **Bob**: 87

**Charlie** scored the highest with a score of **95**.

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [37]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

/tmp/ipykernel_2689/1261841246.py:2: UserWarning: Agents usage is experimental and may change in future versions.
  my_agent = client.agents.create(


✓ Agent created: my-data-analyst-285a396d


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [38]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the complete Python solution using **pandas** to generate the dataset, perform the revenue analysis, and visualize the findings.

---

### Python Code

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Set random seed for reproducibility
np.random.seed(42)

# 2. Define product catalog and regions
products_catalog = {
    'Laptop': 950.0,
    'Smartphone': 700.0,
    'Tablet': 450.0,
    'Monitor': 300.0,
    'Smartwatch': 250.0,
    'Headphones': 150.0,
    'Camera': 600.0,
    'Printer': 200.0,
    'Keyboard': 80.0,
    'Mouse': 40.0
}

product_names = list(products_catalog.keys())
regions = ['North', 'South', 'East', 'West']
n_records = 100

# 3. Generate 100 sales transactions
chosen_products = np.random.choice(product_names, size=n_records)
chosen_regions = np.random.choice(regions, size=n_records)
quantities = np.random.randint(1, 15, size=n_records)

# Minor pricing variance (+/- 5%) to simulate discounts/promotions
revenues = [
    round(quantities[i] * products_catalog[chosen_products[i]] * np.random.uniform(0.95, 1.05), 2)
    for i in range(n_records)
]

# 4. Construct DataFrame
df = pd.DataFrame({
    'product': chosen_products,
    'region': chosen_regions,
    'revenue': revenues,
    'quantity': quantities
})

# Save raw dataset
df.to_csv('sales_data.csv', index=False)

# 5. Aggregate and identify Top 5 Products by Total Revenue
product_analysis = df.groupby('product').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum'),
    order_count=('quantity', 'count'),
    avg_order_value=('revenue', 'mean')
).reset_index()

top_products = product_analysis.sort_values(by='total_revenue', ascending=False)
top_5 = top_products.head(5).copy()
top_5['revenue_share_pct'] = (top_5['total_revenue'] / df['revenue'].sum()) * 100

# 6. Regional breakdown of the Top 5 Products
top_5_names = top_5['product'].tolist()
top_5_regional = df[df['product'].isin(top_5_names)].pivot_table(
    index='product',
    columns='region',
    values='revenue',
    aggfunc='sum',
    fill_value=0
).loc[top_5_names]

# 7. Visualization 1: Top 5 Products by Revenue Bar Chart
plt.figure(figsize=(9, 5))
bars = plt.barh(top_5['product'][::-1], top_5['total_revenue'][::-1] / 1000, color='#2b5c8f', edgecolor='black', alpha=0.85)
plt.title('Top 5 Products by Total Revenue', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Total Revenue ($ in Thousands)', fontsize=11)
plt.ylabel('Product', fontsize=11)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.6, bar.get_y() + bar.get_height() / 2, f"${w:.1f}k", ha='left', va='center', fontweight='bold')
plt.xlim(0, max(top_5['total_revenue'] / 1000) * 1.15)
plt.tight_layout()
plt.savefig('top_5_products_revenue.png', dpi=300)
plt.close()

# 8. Visualization 2: Regional Revenue Breakdown
top_5_regional.plot(kind='bar', figsize=(10, 6), colormap='viridis', edgecolor='black', alpha=0.85)
plt.title('Regional Revenue Breakdown for Top 5 Products', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Product', fontsize=11)
plt.ylabel('Revenue ($)', fontsize=11)
plt.xticks(rotation=0)
plt.legend(title='Region', frameon=True)
plt.tight_layout()
plt.savefig('top_5_products_by_region.png', dpi=300)
plt.close()
```

---

### Execution Output

#### 1. Sample Dataset (First 10 of 100 records)
```
      product region  revenue  quantity
0      Camera   West   575.59         1
1     Monitor  South   935.75         3
2     Printer  North  2080.08        10
3  Smartwatch   West  3039.93        12
4      Camera   East  4722.73         8
5       Mouse   East   433.37        11
6      Tablet  South  2761.01         6
7      Camera   West  4990.61         8
8     Printer  North  1869.68         9
9  Smartwatch   East  1027.99         4
```

#### 2. Summary Statistics (Overall Dataset)
```
            revenue    quantity
count    100.000000  100.000000
mean    2237.511100    6.440000
std     2433.383029    4.033496
min       39.300000    1.000000
25%      555.927500    3.000000
50%     1208.585000    6.000000
75%     3150.145000   10.000000
max    11871.090000   14.000000
```
- **Total Revenue across 100 transactions**: $223,751.11
- **Total Units Sold**: 644 units

---

### Top 5 Products by Total Revenue

| Rank | Product | Total Revenue | Units Sold | Order Count | Avg Order Value | Revenue Share |
|:----:|:--------|:--------------|:----------:|:-----------:|:---------------:|:-------------:|
| **1** | **Laptop** | **$51,701.85** | 54 | 7 | $7,385.98 | 23.11% |
| **2** | **Smartphone** | **$48,606.50** | 70 | 10 | $4,860.65 | 21.72% |
| **3** | **Camera** | **$26,694.63** | 44 | 11 | $2,426.78 | 11.93% |
| **4** | **Tablet** | **$22,699.29** | 49 | 9 | $2,522.14 | 10.14% |
| **5** | **Printer** | **$21,746.75** | 108 | 15 | $1,449.78 | 9.72% |

#### Regional Breakdown for Top 5 Products ($ Revenue)
```
region          East     North     South      West
product                                           
Laptop      21697.61  23254.51   6749.73      0.00
Smartphone  17054.14   5477.23  14155.76  11919.37
Camera       7736.59   5551.05   5442.27   7964.72
Tablet      13458.47   1402.65   3205.06   4633.11
Printer         0.00  14336.43   4119.86   3290.46
```

---

### Key Analytical Takeaways

1. **Revenue Concentration**:
   - The top two products (**Laptop** at $51,701.85 and **Smartphone** at $48,606.50) generate **44.83%** of all revenue.
   - The top 5 products together account for **$171,449.02**, representing **76.62%** of the company's total revenue across all 10 catalog items.
2. **Volume vs. Price Drivers**:
   - **Laptops** drove the highest revenue despite only having 7 orders (54 units), driven by a high average order value ($7,385.98).
   - **Printers** achieved top-5 ranking primarily through sheer transaction volume and units sold (108 units across 15 orders).
3. **Regional Dynamics**:
   - **North** and **East** were the dominant revenue drivers for high-ticket items like Laptops ($23.2k and $21.7k respectively).
   - **Smartphone** sales showed the most balanced geographic distribution across all four regions.

---

### Generated Output Files
- `sales_data.csv` — Full 100-record dataset.
- `top_5_products_revenue.png` — Horizontal bar chart displaying total revenue for the top 5 products.
- `top_5_products_by_region.png` — Grouped bar chart comparing regional sales across the top 5 products.

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [39]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent-285a396d


In [40]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

I specialize in building applications with the **Gemini API**, leveraging three specialized skills: **`gemini-api-dev`**, **`gemini-live-api-dev`**, and **`gemini-omni-flash-api`**. 

Here is what I can build and accomplish across these areas:

---

### 1. Gemini Interactions API & Agents (`gemini-api-dev`)
Using modern SDKs (`google-genai` for Python and `@google/genai` for TypeScript) and the unified Interactions API, I can build:

- **Multimodal Intelligence & Reasoning:**
  - Build chat and agent applications using `gemini-3.8-flash` (balanced multimodal tasks), `gemini-3.5-flash-lite`, and `gemini-3.1-pro-preview` (deep reasoning and coding).
  - Process video, high-resolution imagery, complex audio recordings, and multi-page documents (PDFs, docs) with context caching and media resolution control.
- **Stateful & Streaming Interactions:**
  - Build persistent multi-turn conversations using server-managed interaction state (`previous_interaction_id`).
  - Implement granular SSE streaming handling typed steps (`step.start`, `step.delta`, `thought`, `step.stop`, `interaction.completed`).
- **Autonomous & Managed Agents:**
  - **Antigravity Agent (`antigravity-preview-09-2026`):** General-purpose managed agents running in sandboxed Linux environments with code execution (Python, Node.js, Bash), file management, and web browsing.
  - **Deep Research Agents (`deep-research-preview-04-2026` / `deep-research-max-preview-04-2026`):** Asynchronous, comprehensive web research with collaborative planning, file search, and MCP integration.
  - **Custom Sandboxed Agents:** Forkable, isolated agent environments configured via `client.agents.create()`.
- **Image Generation & Conversational Editing:**
  - Generate and edit images with `gemini-nano-banana-2.1` (supporting multi-image reference fusion with up to 14 images), `gemini-3-pro-image`, and `gemini-3.1-flash-lite-image`.
- **Expressive Speech (TTS) & Voice Design:**
  - Synthesize speech using `gemini-3.8-flash-tts` and `gemini-3.8-flash-lite-tts` with multi-speaker dialogue, custom Voice Design personas, Voice Replication, and inline vocal annotations.
  - Transcribe audio with `gemini-3.5-transcribe` (verbatim or smart cleaned formatting).
- **Tools, Grounding & Structured Outputs:**
  - Strict JSON schema outputs, Google Search grounding, code execution, URL context, File Search, and custom function calling.

---

### 2. Real-Time Streaming & Gemini Live API (`gemini-live-api-dev`)
Using low-latency, bidirectional WebSocket connections, I can create real-time voice, video, and screen-sharing applications:

- **Low-Latency Voice Assistants (`gemini-3.8-live`):**
  - Bidirectional mic-to-speaker conversational audio (16kHz PCM in, 24kHz PCM out) with simultaneous transcription and natural interruption handling.
- **Extended Thinking Voice Agents (`gemini-3.8-live-extended-thinking`):**
  - Real-time agents that reason over complex tasks and invoke non-blocking asynchronous tools in the background while speaking natural conversational fillers (e.g., *"Checking flight availability now..."*).
- **Live Video & Screen Interaction:**
  - Stream camera feeds or screen captures alongside microphone audio for real-time visual assistance.
- **Live Streaming Transcription (`gemini-3.5-transcribe-live`):**
  - Real-time speech-to-text with interim hypotheses and finalized transcripts, smart formatting, and client-side Hybrid VAD (`audio_stream_end`).
- **Live Speech-to-Speech Translation (`gemini-3.5-live-translate-preview`):**
  - Low-latency real-time voice translation across 70+ languages.
- **Production Architecture & Security:**
  - Generate client-side ephemeral tokens (no API keys exposed in browsers/mobile apps), handle context window compression for long sessions, session resumption, and integrate with platforms like LiveKit or Pipecat.

---

### 3. Generative Video with Gemini Omni Flash (`gemini-omni-flash-api`)
Using `gemini-omni-1.1-flash` and automated workflows with `ffmpeg` and the Files API, I can build generative video pipelines:

- **Text-to-Video Generation:**
  - Generate videos from text prompts with customizable resolutions (`360p`, `720p`, `1080p`, up to `4k`), aspect ratios (`16:9` and `9:16`), and durations (3–10s).
- **Image-to-Video & Transitions:**
  - First-frame-to-video (`<FIRST_FRAME>`) animating static images.
  - Smooth interpolation between first and last frames (`<FIRST_FRAME>` and `<LAST_FRAME>`) or seamless infinite loops.
- **Video Extensions & Turn-by-Turn Generation:**
  - Extend existing generated or uploaded clips by up to 10 seconds per turn, reaching total durations up to 40 seconds.
- **Video Editing & Style Transfer:**
  - Perform video-to-video transformations (e.g., anime styling, object addition, background replacement) while preserving original audio or regenerating full sound design from scratch (`--strip-audio`).
- **Multimodal Reference Guidance:**
  - Direct character, motion, or aesthetic styles using reference images (`<IMAGE_REF_N>`) and reference videos (`<VIDEO_REF_N>`).
- **Batch Processing & Tooling:**
  - Automated inspection (`inspect_video.py`), normalization/trimming (`prep_video.py`), and concurrent parallel generation (`generate_video.py --batch`).

---

### What would you like to build?
Whether you need:
1. A **full-stack agentic web app** (Next.js, Vite, Python FastAPI/Flask),
2. A **real-time voice/video companion** or translation tool,
3. An **automated video generation pipeline**, or
4. Migration of existing code to the latest Gemini models and SDKs,

Let me know what you'd like to create!

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [42]:
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent="my-gemini-api-agent",
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

BadRequestError: Error code: 400 - {'error': {'message': 'Request contains an invalid argument.', 'code': 'invalid_request'}}

### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [ ]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id="my-data-analyst")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

In [ ]:
# Clean up: delete the agents you created.
for agent_name in ["my-data-analyst", "my-forked-agent", "my-gemini-api-agent"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [ ]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [ ]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": [{"x-goog-api-key": GEMINI_API_KEY}],
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [ ]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

In [ ]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations